<div style="background:linear-gradient(135deg,#51247a,#7a3fa0);padding:24px 28px;border-radius:10px;margin-bottom:.8rem;border-bottom:4px solid #f0a500;"><div style="font-size:2.4rem;margin-bottom:6px;">💬</div><div style="color:white;font-size:1.5rem;font-weight:700;">Sentiment Explorer</div><div style="color:rgba(255,255,255,.82);font-size:.92rem;margin-top:4px;">Score texts for polarity and eight basic emotions<br><a href="https://ladal.edu.au/sentiment.html" target="_blank" style="color:#f0c060;font-size:.85rem;">&#x2192; Open the full tutorial</a></div></div>

<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">How to use this tool:</b><ol style="margin:.5rem 0 0 0;padding-left:1.3em;line-height:1.9;"><li>Run <b>Step 1</b> to set up the environment.</li><li>Upload your <code>.txt</code> files to the <b>MyTexts</b> folder, then run <b>Step 2</b>.</li><li>In <b>Step 3</b>, click <b>Run Analysis</b> — no configuration needed.</li><li>Download results from the <b>MyOutput</b> folder.</li></ol></div>

<div style="background:#fff8e1;border-left:4px solid #f0a500;border-radius:6px;padding:12px 16px;margin:.6rem 0;font-family:sans-serif;font-size:.88rem;">&#x1F4D6; <b>This notebook contains code cells.</b> Do not edit the code — just run each step using the <b>Run Step</b> buttons. To run a cell: click on it and press <b>Shift + Enter</b>, or use the <b>&#x25B6; Run</b> button in the toolbar above.</div>

<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.8rem;opacity:.75;">STEP 1</span><br><b style="font-size:1rem;">&#x2699;&#xFE0F; Set up the environment — run this first</b></div>

<div style="background:#f4f0f8;border-left:4px solid #51247a;border-radius:6px;padding:12px 16px;margin:.6rem 0;font-family:sans-serif;font-size:.88rem;"><b>Run the cell below once</b> to load all the helper functions this notebook needs. Click on the cell, then press <b>Shift + Enter</b>. You should see a green <em>Setup complete</em> message when it finishes.</div>

<div style="background:#f0f0f0;border-left:3px solid #aaa;padding:4px 12px;margin-bottom:2px;font-size:.78rem;color:#888;font-family:sans-serif;">&#x1F512; <em>Step 1 — environment setup — run this cell, do not edit it</em></div>

In [ ]:
# Setup: load display helpers and shared functions
# Run this cell once — then proceed to the steps below.

suppressPackageStartupMessages(library(IRdisplay))
suppressPackageStartupMessages(library(IRkernel))
suppressPackageStartupMessages(library(ipywidgets))

# ── Colour-coded feedback boxes ──────────────────────────────────────
.box <- function(msg, bg, border, icon="") {
  IRdisplay::display_html(paste0(
    "<div style=\"background:", bg,
    ";border-left:4px solid ", border,
    ";border-radius:6px;padding:10px 15px;margin:.4rem 0;",
    "font-family:sans-serif;font-size:.9rem;\">",
    if (nzchar(icon)) paste0(icon, " ") else "",
    msg, "</div>"))
}
ok   <- function(msg) .box(msg, "#eafaf1", "#27ae60", "&#x2705;")
warn <- function(msg) .box(msg, "#fff8e1", "#f0a500", "&#x26A0;&#xFE0F;")
err  <- function(msg) .box(msg, "#fff0f0", "#e74c3c", "&#x274C;")
info <- function(msg) .box(msg, "#f4f0f8", "#51247a", "&#x2139;&#xFE0F;")

# ── Progress bar ─────────────────────────────────────────────────────
.prog <- function(label, value, max=100) {
  pct <- round(100 * value / max)
  IRdisplay::display_html(paste0(
    "<div style=\"font-family:sans-serif;font-size:.85rem;margin:.3rem 0;\">",
    "<span style=\"color:#51247a;font-weight:600;\">", label, "</span>",
    "<div style=\"background:#e8e4f0;border-radius:4px;height:8px;margin-top:3px;\">",
    "<div style=\"background:#51247a;width:", pct,
    "%;height:8px;border-radius:4px;\"></div></div></div>"))
}

# ── Load plain-text files from a folder ──────────────────────────────
load_texts <- function(folder = "notebooks/MyTexts") {
  files <- list.files(folder, pattern = "\\.txt$",
                      full.names = TRUE, recursive = FALSE)
  if (length(files) == 0)
    stop("No .txt files found in ", folder,
         ". Please upload your files first.")
  texts <- vapply(files, function(f)
    paste(readLines(f, warn = FALSE, encoding = "UTF-8"), collapse = " "),
    character(1))
  names(texts) <- tools::file_path_sans_ext(basename(files))
  texts
}

# ── Save texts to MyOutput ───────────────────────────────────────────
save_texts <- function(texts, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  for (nm in names(texts))
    writeLines(texts[[nm]], file.path(folder, paste0(nm, ".txt")))
}

# ── Save data frame as Excel ─────────────────────────────────────────
save_excel <- function(df, filename, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  writexl::write_xlsx(as.data.frame(df), file.path(folder, filename))
}

ok("Setup complete. Follow the steps below.")


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.8rem;opacity:.75;">STEP 2</span><br><b style="font-size:1rem;">&#x1F4C2; Upload your texts to the MyTexts folder</b></div>

<div style="background:#f4f0f8;border:2px dashed #51247a;border-radius:8px;padding:18px 22px;margin:.6rem 0;font-family:sans-serif;"><b style="color:#51247a;font-size:.95rem;">&#x1F4C2; Upload your files</b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;font-size:.9rem;line-height:1.9;"><li>Look at the <b>file browser panel</b> on the left side of the screen.</li><li>Double-click the <b><code>MyTexts</code></b> folder to open it.</li><li><b>Drag and drop</b> your <code>.txt</code> files into that folder.</li><li>When done, click the <b>Run Step</b> button below.</li></ol><p style="margin:.5rem 0 0 0;font-size:.82rem;color:#888;">Only <code>.txt</code> files are accepted. You can upload multiple files at once.</p></div>

<div style="background:#f0f0f0;border-left:3px solid #aaa;padding:4px 12px;margin-bottom:2px;font-size:.78rem;color:#888;font-family:sans-serif;">&#x1F512; <em>Step 2 — load uploaded files — run this cell, do not edit it</em></div>

In [ ]:
# Run this cell to load your uploaded files.
btn_load <- ipywidgets::Button(
  description = "  Run Step: Load Files",
  button_style = "info", icon = "folder-open",
  layout = ipywidgets::Layout(width="220px", height="36px"))
out_load <- ipywidgets::Output()
ipywidgets::display(ipywidgets::VBox(list(btn_load, out_load)))

ipywidgets::observe(btn_load, "clicks", function(change) {
  ipywidgets::with_output(out_load, {
    tryCatch({
      texts <<- load_texts("notebooks/MyTexts")
      ok(paste0("Loaded <b>", length(texts),
                " file(s)</b>: ",
                paste(names(texts), collapse=", ")))
    }, error=function(e) err(conditionMessage(e)))
  })
})


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.8rem;opacity:.75;">STEP 3</span><br><b style="font-size:1rem;">&#x1F4CA; Configure and run the analysis</b></div>

<div style="background:#f4f0f8;border-left:4px solid #51247a;border-radius:6px;padding:12px 16px;margin:.6rem 0;font-family:sans-serif;font-size:.88rem;">Set your options using the controls below, then click the green <b>Run Analysis</b> button. Results will appear directly below the button and will also be saved to the <b>MyOutput</b> folder.</div>

<div style="background:#f0f0f0;border-left:3px solid #aaa;padding:4px 12px;margin-bottom:2px;font-size:.78rem;color:#888;font-family:sans-serif;">&#x1F512; <em>Step 3 — analysis — run this cell, do not edit it</em></div>

In [ ]:
# Click Run Analysis to score your texts.
suppressPackageStartupMessages(library(syuzhet))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(tidyr))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(writexl))

w_btn <- ipywidgets::Button(
  description="  Run Analysis",
  button_style="success", icon="heart",
  layout=ipywidgets::Layout(width="180px", height="38px"))
w_out <- ipywidgets::Output()
ipywidgets::display(ipywidgets::VBox(list(w_btn, w_out)))

emo_cols <- c("anger","anticipation","disgust","fear",
              "joy","sadness","surprise","trust")

ipywidgets::observe(w_btn, "clicks", function(change) {
  ipywidgets::with_output(w_out, {
    tryCatch({
      if (!exists("texts")) stop("No files loaded. Run Step 2 first.")
      senti_list <- lapply(seq_along(texts), function(i) {
        nm <- names(texts)[i]
        .prog(paste0("Scoring: ", nm, " (", i, "/", length(texts), ")"),
              5 + 70*(i/length(texts)))
        words  <- tolower(unlist(strsplit(texts[[nm]], "[^a-z]+")))
        words  <- words[nzchar(words)]
        if (length(words)==0) return(NULL)
        scores <- syuzhet::get_nrc_sentiment(words)
        scores$word <- words; scores$document <- nm
        scores
      })
      senti <- dplyr::bind_rows(senti_list)
      .prog("Summarising by document...", 80)
      senti_sum <- senti |>
        dplyr::group_by(document) |>
        dplyr::summarise(
          words=n(),
          dplyr::across(dplyr::all_of(c(emo_cols,"negative","positive")), sum),
          sentiment=positive-negative, .groups="drop")
      print(senti_sum)
      .prog("Plotting emotion profiles...", 88)
      emo_pal <- c(anger="#e74c3c",anticipation="#f39c12",disgust="#8e44ad",
                   fear="#2c3e50",joy="#27ae60",sadness="#3498db",
                   surprise="#1abc9c",trust="#d35400")
      p <- senti_sum |>
        tidyr::pivot_longer(dplyr::all_of(emo_cols),
                            names_to="emotion", values_to="count") |>
        ggplot(aes(x=emotion, y=count, fill=emotion)) +
        geom_col(width=.7, show.legend=FALSE) +
        facet_wrap(~document, scales="free_y") +
        scale_fill_manual(values=emo_pal) +
        coord_flip() + theme_bw(base_size=11) +
        labs(title="Emotion profile by document", x=NULL, y="Word count")
      print(p)
      .prog("Saving...", 95)
      save_excel(senti,     "sentiment_words.xlsx")
      save_excel(senti_sum, "sentiment_summary.xlsx")
      dir.create("notebooks/MyOutput", showWarnings=FALSE, recursive=TRUE)
      ggsave("notebooks/MyOutput/sentiment_plot.png",
             bg="white", width=9, height=5, dpi=200)
      .prog("Done.", 100)
      ok("Saved <b>sentiment_words.xlsx</b>, <b>sentiment_summary.xlsx</b>, <b>sentiment_plot.png</b>.")
    }, error=function(e) err(conditionMessage(e)))
  })
})


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.8rem;opacity:.75;">STEP 99</span><br><b style="font-size:1rem;">&#x2B07;&#xFE0F; Download your results</b></div>

<div style="background:#f4f0f8;border-left:4px solid #51247a;border-radius:6px;padding:12px 16px;margin:.6rem 0;font-family:sans-serif;font-size:.88rem;">Your results have been saved to the <b>MyOutput</b> folder. To download: open the <b>file browser panel</b> on the left, double-click <b>MyOutput</b>, then <b>right-click</b> on a file and choose <b>Download</b>.</div>

---

### Citation

Schweinberger, Martin. (2025). *LADAL Sentiment Explorer*. Brisbane: The University of Queensland. <https://ladal.edu.au/tools.html>

```bibtex
@misc{schweinberger2025sentiment,
  author = {Schweinberger, Martin},
  title  = {LADAL Sentiment Explorer},
  year   = {2025},
  organization = {The University of Queensland},
  url    = {https://ladal.edu.au/tools.html}
}
```


In [ ]:
sessionInfo()